In [ ]:
# Example 1: Top-level public API surface — verify the headline exports exist and are classes.
import inspect

from langchain_google_vertexai import (
    ChatVertexAI,
    VertexAI,
    VertexAIEmbeddings,
    HarmBlockThreshold,
    HarmCategory,
)

exports = {
    "ChatVertexAI": ChatVertexAI,
    "VertexAI": VertexAI,
    "VertexAIEmbeddings": VertexAIEmbeddings,
    "HarmBlockThreshold": HarmBlockThreshold,
    "HarmCategory": HarmCategory,
}

for name, obj in exports.items():
    print(f"{name}: {type(obj).__name__}")
    assert obj is not None

assert inspect.isclass(ChatVertexAI)
assert inspect.isclass(VertexAI)
assert inspect.isclass(VertexAIEmbeddings)

In [ ]:
# Example 2: Safety setting enums — verify HarmCategory / HarmBlockThreshold are usable enums.
from langchain_google_vertexai import HarmBlockThreshold, HarmCategory

# These come from google.cloud.aiplatform_v1beta1.types — they're proto enums.
categories = list(HarmCategory)
thresholds = list(HarmBlockThreshold)

print(f"HarmCategory has {len(categories)} members")
print(f"HarmBlockThreshold has {len(thresholds)} members")

# Show a few names to sanity-check.
for c in categories[:4]:
    print(f"  HarmCategory.{c.name}")
for t in thresholds[:4]:
    print(f"  HarmBlockThreshold.{t.name}")

# Build a safety-settings dict like a real caller would.
safety_settings = {c: HarmBlockThreshold.BLOCK_ONLY_HIGH for c in categories}
assert len(safety_settings) == len(categories)
assert len(categories) > 0
assert len(thresholds) > 0

In [ ]:
# Example 3: Inspect ChatVertexAI's pydantic schema without instantiating (no GCP auth required).
from langchain_google_vertexai import ChatVertexAI
from langchain_core.language_models.chat_models import BaseChatModel

# Class metadata sanity checks.
assert issubclass(ChatVertexAI, BaseChatModel)

# Pydantic v2 model_fields exposes the configurable parameters.
fields = ChatVertexAI.model_fields
expected_fields = {"model_name", "temperature", "max_output_tokens", "project", "location"}
present = expected_fields & set(fields.keys())
print(f"ChatVertexAI fields present: {sorted(present)}")
print(f"Total ChatVertexAI fields: {len(fields)}")

assert "model_name" in fields or "model" in fields, "expected a model selector field"
assert "temperature" in fields

In [ ]:
# Example 4: Function/tool calling — verify _format_tools_to_vertex_tool can convert a Python callable.
from langchain_core.tools import tool
from langchain_google_vertexai.functions_utils import _format_to_gapic_tool

@tool
def get_weather(city: str) -> str:
    """Return the weather for a given city."""
    return f"sunny in {city}"

# Convert the LangChain tool into a Vertex-AI gapic Tool proto.
gapic_tool = _format_to_gapic_tool([get_weather])

print(f"gapic_tool type: {type(gapic_tool).__name__}")
func_decls = gapic_tool.function_declarations
print(f"function_declarations: {len(func_decls)}")
for fd in func_decls:
    print(f"  - {fd.name}: {fd.description!r}")

assert len(func_decls) == 1
assert func_decls[0].name == "get_weather"

In [ ]:
# Example 5: Embeddings model class introspection — no network call.
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_core.embeddings import Embeddings

assert issubclass(VertexAIEmbeddings, Embeddings)

# The class must declare embed_documents + embed_query (the Embeddings contract).
assert hasattr(VertexAIEmbeddings, "embed_documents")
assert hasattr(VertexAIEmbeddings, "embed_query")

fields = VertexAIEmbeddings.model_fields
print(f"VertexAIEmbeddings field count: {len(fields)}")
for name in sorted(fields.keys())[:10]:
    print(f"  - {name}")

# Final assert: package's headline classes all loaded and have the correct base.
from langchain_google_vertexai import ChatVertexAI, VertexAI
from langchain_core.language_models.llms import BaseLLM

assert issubclass(VertexAI, BaseLLM)
print("\nlangchain_google_vertexai smoke tests passed")